In [1]:
import requests
import pandas as pd
import json
import datetime as dt

In [3]:
def binance_data(symbol, interval, startTime, endTime, chunk_size=1500):
    url = "https://api.binance.com/api/v3/klines"
    df_list = []
    
    while startTime < endTime:
        # Calculate the end time for the current chunk (1 minute)
        end_time = startTime + pd.DateOffset(minutes=1)
        end_time = min(end_time, endTime)
        
        start_time_ms = int(startTime.timestamp() * 1000)
        end_time_ms = int(end_time.timestamp() * 1000)
        
        req_params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": str(start_time_ms),
            "endTime": str(end_time_ms),
            "limit": str(chunk_size)
        }
        
        response = requests.get(url, params=req_params)
        if response.status_code == 200:
            data = json.loads(response.text)
            if len(data) > 0:
                df = pd.DataFrame(data)
                df = df.iloc[:, 0:6]
                df.columns = ["opentime", "open", "high", "low", "close", "volume"]
                df.index = [dt.datetime.fromtimestamp(x / 1000.0) for x in df.opentime]
                df.drop(columns=['opentime'], inplace=True)
                df.insert(0, "time", df.index)
                df_list.append(df)
        
        startTime = end_time
    
    if not df_list:
        return None
    
    final_df = pd.concat(df_list)
    return final_df

In [6]:
df = binance_data("BTCUSDT","1m",dt.datetime(2023,8,9),dt.datetime(2023,8,10))



In [7]:
df

,time,open,high,low,close,volume
2023-08-09 00:00:00,2023-08-09 00:00:00,29880.77000000,29905.94000000,29878.01000000,29905.93000000,50.21977000
2023-08-09 00:01:00,2023-08-09 00:01:00,29905.94000000,29913.48000000,29880.00000000,29894.02000000,92.84983000
2023-08-09 00:02:00,2023-08-09 00:02:00,29894.01000000,29894.02000000,29866.83000000,29866.84000000,40.21649000
2023-08-09 00:03:00,2023-08-09 00:03:00,29866.83000000,29876.00000000,29862.31000000,29865.24000000,71.25239000
2023-08-09 00:04:00,2023-08-09 00:04:00,29865.23000000,29865.24000000,29832.45000000,29834.01000000,128.13230000
...,...,...,...,...,...,...
2023-08-10 05:28:00,2023-08-10 05:28:00,29588.00000000,29588.00000000,29566.49000000,29578.24000000,47.24313000
2023-08-10 05:28:00,2023-08-10 05:28:00,29588.00000000,29588.00000000,29566.49000000,29578.24000000,47.24313000
2023-08-10 05:29:00,2023-08-10 05:29:00,29578.24000000,29582.00000000,29578.24000000,29581.99000000,9.21946000
2023-08-10 05:29:00,2023-08-10 05:29:00,29578.24000000,29582.00000000,29578.24000000,29581.99000000,9.21946000


In [8]:
df.to_csv("btcdataf.csv",index=False)